# Sentinel-2 & Hansen Data Fetching from Google Earth Engine

**Study Area:** East Kalimantan, Indonesia  
**Period:** 2021-2024  

---

## Details

| Methodology | Description |
|-------|----------|
| Band export strategy | Single multi-band export (B2, B3, B4, B8) + compute indices locally |
| Hansen resolution | Exported at 10m with nearest-neighbor resampling to match S2 grid |
| Hansen band naming | Explicit `.rename()` after Boolean operations |
| Hansen change mask | Proper bitemporal change mask: loss between year T1 and T2 |
| Nodata handling | Explicit `unmask(0)` for ground truth, proper nodata for S2 |
| Tile fragmentation | Use `fileDimensions` parameter to control tiling, or merge post-download |
| EVI scaling | All bands exported as Float32 or computed locally from raw bands |
| Cloud masking | SCL also exported separately for post-hoc quality control |

In [1]:
import ee
import numpy as np

In [2]:
ee.Authenticate()
ee.Initialize(project='tugas-akhir-485606')

## 1. Define Study Area

In [3]:
kalimantan_roi = ee.Geometry.Rectangle([
    116.0,  # west
    -1.5,   # south
    117.5,  # east
    -0.2    # north
])

roi = kalimantan_roi
years = [2021, 2022, 2023, 2024]

print(f"Study Area: {roi.area().divide(1e6).getInfo():.2f} km2")
print(f"Years: {years}")

Study Area: 24108.67 km2
Years: [2021, 2022, 2023, 2024]


## 2. Create Sentinel-2 Composites

### Key Change: Export Raw Bands Together

Instead of exporting NGB, RGB, NDVI, EVI separately (which creates different tile grids
and wastes storage), we export the four raw 10m bands (B2, B3, B4, B8) as a single
multi-band GeoTIFF per year. All indices (NDVI, EVI, NGB) can then be computed
locally during preprocessing, guaranteeing perfect spatial alignment.

In [6]:
def create_sentinel2_composite(roi, year, cloud_threshold=20):
    """
    Create a yearly median composite of Sentinel-2 SR imagery.

    Exports the four 10m-resolution bands needed to derive all
    band combinations used in the paper:
        - NGB  = [B8, B3, B2]
        - NDVI = (B8 - B4) / (B8 + B4)
        - EVI  = 2.5 * (B8 - B4) / (B8 + 6*B4 - 7.5*B2 + 1)

    Cloud masking uses the SCL band (Scene Classification Layer),
    consistent with the paper methodology.

    Parameters
    ----------
    roi : ee.Geometry
        Region of interest.
    year : int
        Calendar year for the composite.
    cloud_threshold : int
        Maximum CLOUDY_PIXEL_PERCENTAGE metadata filter.

    Returns
    -------
    ee.Image
        Median composite with bands [B2, B3, B4, B8] clipped to ROI.
    """
    start = ee.Date.fromYMD(year, 1, 1)
    end = start.advance(1, 'year')

    s2 = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(roi)
        .filterDate(start, end)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_threshold))
    )

    count = s2.size().getInfo()
    print(f"Year {year}: {count} images available")

    if count == 0:
        raise RuntimeError(
            f"No images found for {year}. "
            "Consider raising cloud_threshold or narrowing the ROI."
        )

    def mask_clouds_scl(image):
        """Mask clouds, cloud shadows, and saturated/defective pixels via SCL."""
        scl = image.select('SCL')
        # SCL classes to mask:
        #   1 = Saturated/Defective
        #   3 = Cloud Shadow
        #   8 = Cloud Medium Probability
        #   9 = Cloud High Probability
        #  10 = Thin Cirrus
        mask = (
            scl.neq(1)
            .And(scl.neq(3))
            .And(scl.neq(8))
            .And(scl.neq(9))
            .And(scl.neq(10))
        )
        return image.updateMask(mask)

    # Select only the 10m bands we need, then composite
    bands_10m = ['B2', 'B3', 'B4', 'B8']
    composite = (
        s2
        .map(mask_clouds_scl)
        .select(bands_10m)
        .median()
        .clip(roi)
    )

    return composite.set('year', year)


composites = {}
for year in years:
    threshold = 35 if year == 2022 else 20
    composites[year] = create_sentinel2_composite(roi, year, cloud_threshold=threshold)

print("\nAll composites created.")

Year 2021: 25 images available
Year 2022: 26 images available
Year 2023: 24 images available
Year 2024: 29 images available

All composites created.


## 3. Hansen Ground Truth

### Key Changes

1. **Export at 10m** to match the Sentinel-2 pixel grid (nearest-neighbor resampling
   is appropriate for a categorical/binary mask).  
2. **Explicit band renaming** after Boolean operations to avoid ambiguous band names.  
3. **Bitemporal change mask**: For ChangeFormer, we need a binary mask indicating
   where deforestation occurred *between* year T1 and year T2. We construct this
   for each consecutive year pair (2021-2022, 2022-2023, 2023-2024).  
4. **`unmask(0)`** to fill masked pixels with 0 (no change), preventing nodata gaps
   in the ground truth.

In [7]:
def get_hansen_data(roi):
    """
    Retrieve Hansen Global Forest Change v1.12 layers.

    Returns
    -------
    dict with keys:
        'forest2000'     : ee.Image  -- binary mask of forest in 2000 (>30% canopy)
        'lossyear'       : ee.Image  -- year of loss (1-24 for 2001-2024, 0 = no loss)
        'annual_loss'    : dict[int, ee.Image]  -- annual loss mask per year
        'bitemporal_change' : dict[str, ee.Image]  -- change between consecutive years
    """
    gfc = ee.Image('UMD/hansen/global_forest_change_2024_v1_12')

    # Forest baseline: >30% tree canopy cover in 2000
    forest2000 = gfc.select('treecover2000').gt(30).rename('forest2000')

    # Loss year band
    lossyear = gfc.select('lossyear')

    # --- Annual loss masks ---
    # For each year, binary mask: 1 = forest lost in that year, 0 = otherwise
    annual_loss = {}
    for year in years:
        year_code = year - 2000  # e.g., 2021 -> 21
        loss_mask = (
            forest2000
            .And(lossyear.eq(year_code))
            .rename('loss')
            .unmask(0)
            .clip(roi)
        )
        annual_loss[year] = loss_mask

    # --- Bitemporal change masks ---
    # For ChangeFormer: binary mask of deforestation between T1 and T2.
    # change(T1->T2) = pixels that were forest at T1 but lost during T2.
    # More precisely: pixels where lossyear == T2_code AND had forest in 2000
    # AND were not already lost before T1.
    bitemporal_change = {}
    for i in range(len(years) - 1):
        t1, t2 = years[i], years[i + 1]
        t2_code = t2 - 2000

        # Deforestation that occurred in year T2
        # (i.e., new deforestation visible when comparing T1 composite to T2 composite)
        change_mask = (
            forest2000
            .And(lossyear.eq(t2_code))
            .rename('change')
            .unmask(0)
            .toUint8()
            .clip(roi)
        )
        key = f"{t1}_{t2}"
        bitemporal_change[key] = change_mask
        print(f"  Bitemporal change mask: {t1} -> {t2}")

    return {
        'forest2000': forest2000.clip(roi),
        'lossyear': lossyear.clip(roi),
        'annual_loss': annual_loss,
        'bitemporal_change': bitemporal_change,
    }


print("Preparing Hansen ground truth layers...")
hansen = get_hansen_data(roi)
print("Done.")

Preparing Hansen ground truth layers...
  Bitemporal change mask: 2021 -> 2022
  Bitemporal change mask: 2022 -> 2023
  Bitemporal change mask: 2023 -> 2024
Done.


## 4. Google Drive's Export

### Export strategy

| Dataset | Bands | Scale | Notes |
|---------|-------|-------|-------|
| S2 composite per year | B2, B3, B4, B8 | 10m | All indices derived locally |
| Hansen bitemporal change | change | 10m | Resampled to match S2 grid |
| Hansen annual loss | loss | 10m | For reference/validation |
| Hansen forest2000 | forest2000 | 10m | Baseline reference |

In [8]:
DRIVE_FOLDER = 'GEE_Kalimantan_v2'

def export_to_drive(image, description, bands, scale=10, folder=DRIVE_FOLDER):
    """
    Export an ee.Image to Google Drive.

    Uses fileDimensions to keep tile sizes manageable (max ~16384 pixels per side)
    while preserving spatial alignment across all exports.
    """
    task = ee.batch.Export.image.toDrive(
        image=image.select(bands),
        description=description,
        folder=folder,
        region=roi,
        scale=scale,
        maxPixels=1e13,
        crs='EPSG:4326',
        fileDimensions=16384,  # Controls tile splitting; same for all exports
    )
    task.start()
    print(f"  Export started: {description}")
    return task

In [9]:
tasks = []

# --- Sentinel-2 multi-band composites ---
print("Exporting Sentinel-2 composites...")
for year in years:
    task = export_to_drive(
        composites[year],
        f"Kalimantan_S2_Bands_{year}",
        bands=['B2', 'B3', 'B4', 'B8'],
        scale=10,
    )
    tasks.append(task)

# --- Hansen bitemporal change masks ---
print("\nExporting Hansen bitemporal change masks...")
for key, change_img in hansen['bitemporal_change'].items():
    task = export_to_drive(
        change_img,
        f"Kalimantan_Hansen_Change_{key}",
        bands=['change'],
        scale=10,  # Resample to 10m to match S2 grid
    )
    tasks.append(task)

# --- Hansen annual loss (for reference) ---
print("\nExporting Hansen annual loss masks...")
for year in years:
    task = export_to_drive(
        hansen['annual_loss'][year],
        f"Kalimantan_Hansen_Loss_{year}",
        bands=['loss'],
        scale=10,
    )
    tasks.append(task)

# --- Hansen forest baseline ---
print("\nExporting Hansen forest baseline...")
task = export_to_drive(
    hansen['forest2000'],
    "Kalimantan_Hansen_Forest2000",
    bands=['forest2000'],
    scale=10,
)
tasks.append(task)

print(f"\nTotal {len(tasks)} export tasks submitted.")
print("Monitor progress at: https://code.earthengine.google.com/tasks")

Exporting Sentinel-2 composites...
  Export started: Kalimantan_S2_Bands_2021
  Export started: Kalimantan_S2_Bands_2022
  Export started: Kalimantan_S2_Bands_2023
  Export started: Kalimantan_S2_Bands_2024

Exporting Hansen bitemporal change masks...
  Export started: Kalimantan_Hansen_Change_2021_2022
  Export started: Kalimantan_Hansen_Change_2022_2023
  Export started: Kalimantan_Hansen_Change_2023_2024

Exporting Hansen annual loss masks...
  Export started: Kalimantan_Hansen_Loss_2021
  Export started: Kalimantan_Hansen_Loss_2022
  Export started: Kalimantan_Hansen_Loss_2023
  Export started: Kalimantan_Hansen_Loss_2024

Exporting Hansen forest baseline...
  Export started: Kalimantan_Hansen_Forest2000

Total 12 export tasks submitted.
Monitor progress at: https://code.earthengine.google.com/tasks


## 5. Post-Download: Expected File Layout

```
GEE_Kalimantan_v2/
    Kalimantan_S2_Bands_2021-*.tif       # 4-band (B2,B3,B4,B8) per tile
    Kalimantan_S2_Bands_2022-*.tif
    Kalimantan_S2_Bands_2023-*.tif
    Kalimantan_S2_Bands_2024-*.tif
    Kalimantan_Hansen_Change_2021_2022-*.tif   # bitemporal change mask
    Kalimantan_Hansen_Change_2022_2023-*.tif
    Kalimantan_Hansen_Change_2023_2024-*.tif
    Kalimantan_Hansen_Loss_2021-*.tif          # annual loss reference
    ...
    Kalimantan_Hansen_Forest2000-*.tif          # baseline
```

All files share the same CRS (EPSG:4326), scale (10m), and tile grid
(`fileDimensions=16384`), which guarantees pixel-level alignment during
preprocessing.